In [14]:
# Principal:    alex
# Role:         data_scientist
# Catalog Role: data_scientist_catalog_role
# Привилегии:   NAMESPACE_LIST (catalog)
#               bronze: TABLE_LIST, TABLE_READ_DATA
#               silver: TABLE_LIST, TABLE_READ_DATA
#               gold:   TABLE_LIST, TABLE_READ_DATA
#
# Матрица доступов:
#   bronze: только чтение
#   silver: только чтение
#   gold:   только чтение
#
# FORBIDDEN: INSERT bronze, INSERT silver, INSERT gold, CREATE bronze

In [15]:
import os
from pyspark.sql import SparkSession

client_id = os.environ["ALEX_CLIENT_ID"]
client_secret = os.environ["ALEX_CLIENT_SECRET"]
credential = f"{client_id}:{client_secret}"

spark = SparkSession.builder \
    .appName("lakehouse-alex-rbac") \
    .config("spark.sql.catalog.lakehouse.credential", credential) \
    .getOrCreate()

spark

In [16]:
spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("SHOW TABLES IN lakehouse.bronze").show(truncate=False)
spark.sql("SHOW TABLES IN lakehouse.silver").show(truncate=False)
spark.sql("SHOW TABLES IN lakehouse.gold").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|lakehouse    |
|spark_catalog|
+-------------+

+---------+---------------+-----------+
|namespace|tableName      |isTemporary|
+---------+---------------+-----------+
|bronze   |raw_products   |false      |
|bronze   |raw_customers  |false      |
|bronze   |raw_categories |false      |
|bronze   |raw_order_items|false      |
|bronze   |raw_orders     |false      |
+---------+---------------+-----------+

+---------+-----------+-----------+
|namespace|tableName  |isTemporary|
+---------+-----------+-----------+
|silver   |customers  |false      |
|silver   |products   |false      |
|silver   |orders     |false      |
|silver   |order_items|false      |
+---------+-----------+-----------+

+---------+----------------------+-----------+
|namespace|tableName             |isTemporary|
+---------+----------------------+-----------+
|gold     |mart_sales_by_category|false      |
|gold     |mart_top_customers    |false      |
+---------+-------

In [17]:
print("[ALLOWED] TABLE_LIST — lakehouse.bronze")
spark.sql("SHOW TABLES IN lakehouse.bronze").show(truncate=False)

[ALLOWED] TABLE_LIST — lakehouse.bronze
+---------+---------------+-----------+
|namespace|tableName      |isTemporary|
+---------+---------------+-----------+
|bronze   |raw_products   |false      |
|bronze   |raw_customers  |false      |
|bronze   |raw_categories |false      |
|bronze   |raw_order_items|false      |
|bronze   |raw_orders     |false      |
+---------+---------------+-----------+



In [18]:
print("[ALLOWED] TABLE_READ_DATA — lakehouse.bronze.raw_categories")
spark.sql("""
    SELECT
        id,
        name,
        description
    FROM lakehouse.bronze.raw_categories
    LIMIT 3
""").show(truncate=False)

[ALLOWED] TABLE_READ_DATA — lakehouse.bronze.raw_categories
+---+-----------+---------------------+
|id |name       |description          |
+---+-----------+---------------------+
|1  |Electronics|Электроника и гаджеты|
|2  |Clothing   |Одежда и аксессуары  |
|3  |Books      |Книги и журналы      |
+---+-----------+---------------------+



In [19]:
print("[ALLOWED] TABLE_FULL_METADATA — lakehouse.bronze.raw_orders")
spark.sql("DESCRIBE TABLE EXTENDED lakehouse.bronze.raw_orders").show(truncate=False)

[ALLOWED] TABLE_FULL_METADATA — lakehouse.bronze.raw_orders
+-----------------------------+------------------------------------------------------+-------+
|col_name                     |data_type                                             |comment|
+-----------------------------+------------------------------------------------------+-------+
|id                           |int                                                   |NULL   |
|customer_id                  |int                                                   |NULL   |
|status                       |string                                                |NULL   |
|total_amount                 |decimal(12,2)                                         |NULL   |
|order_date                   |string                                                |NULL   |
|                             |                                                      |       |
|# Metadata Columns           |                                                      

In [20]:
print("[ALLOWED] TABLE_LIST — lakehouse.silver")
spark.sql("SHOW TABLES IN lakehouse.silver").show(truncate=False)

[ALLOWED] TABLE_LIST — lakehouse.silver
+---------+-----------+-----------+
|namespace|tableName  |isTemporary|
+---------+-----------+-----------+
|silver   |customers  |false      |
|silver   |products   |false      |
|silver   |orders     |false      |
|silver   |order_items|false      |
+---------+-----------+-----------+



In [21]:
print("[ALLOWED] TABLE_READ_DATA — lakehouse.silver.customers")
spark.sql("""
    SELECT
        id,
        name,
        email,
        city,
        created_at
    FROM lakehouse.silver.customers
    LIMIT 3
""").show(truncate=False)

[ALLOWED] TABLE_READ_DATA — lakehouse.silver.customers
+---+----------------------------+----------------------------+----------+----------+
|id |name                        |email                       |city      |created_at|
+---+----------------------------+----------------------------+----------+----------+
|1  |Козлова Ирина Ефимовна      |visheslavzinovev@example.org|д. Ребриха|2023-06-14|
|9  |Татьяна Кузьминична Соболева|nikola2019@example.net      |п. Кунгур |2023-10-11|
|14 |Фадеев Гостомысл Ааронович  |selivan_01@example.com      |п. Беслан |2024-01-29|
+---+----------------------------+----------------------------+----------+----------+



In [22]:
print("[ALLOWED] TABLE_FULL_METADATA — lakehouse.silver.orders")
spark.sql("DESCRIBE TABLE EXTENDED lakehouse.silver.orders").show(truncate=False)

[ALLOWED] TABLE_FULL_METADATA — lakehouse.silver.orders
+-----------------------------+--------------------------------------------------+-------+
|col_name                     |data_type                                         |comment|
+-----------------------------+--------------------------------------------------+-------+
|id                           |int                                               |NULL   |
|customer_id                  |int                                               |NULL   |
|status                       |string                                            |NULL   |
|total_amount                 |decimal(12,2)                                     |NULL   |
|order_date                   |date                                              |NULL   |
|                             |                                                  |       |
|# Metadata Columns           |                                                  |       |
|_spec_id                     |int

In [23]:
print("[ALLOWED] TABLE_LIST — lakehouse.gold")
spark.sql("SHOW TABLES IN lakehouse.gold").show(truncate=False)

[ALLOWED] TABLE_LIST — lakehouse.gold
+---------+----------------------+-----------+
|namespace|tableName             |isTemporary|
+---------+----------------------+-----------+
|gold     |mart_sales_by_category|false      |
|gold     |mart_top_customers    |false      |
+---------+----------------------+-----------+



In [24]:
print("[ALLOWED] TABLE_READ_DATA — lakehouse.gold.mart_sales_by_category")
spark.sql("""
    SELECT
        category_name,
        month,
        total_revenue,
        order_count,
        avg_check
    FROM lakehouse.gold.mart_sales_by_category
    LIMIT 3
""").show(truncate=False)

[ALLOWED] TABLE_READ_DATA — lakehouse.gold.mart_sales_by_category
+-------------+-------+-------------+-----------+---------+
|category_name|month  |total_revenue|order_count|avg_check|
+-------------+-------+-------------+-----------+---------+
|Books        |2025-05|9691.97      |106        |91.43    |
|Clothing     |2025-05|30004.05     |86         |348.88   |
|Electronics  |2025-05|249805.97    |104        |2401.98  |
+-------------+-------+-------------+-----------+---------+



In [25]:
print("[ALLOWED] TABLE_FULL_METADATA — lakehouse.gold.mart_sales_by_category")
spark.sql("DESCRIBE TABLE EXTENDED lakehouse.gold.mart_sales_by_category").show(truncate=False)

[ALLOWED] TABLE_FULL_METADATA — lakehouse.gold.mart_sales_by_category
+-----------------------------+----------------------------------------------------------------+-------+
|col_name                     |data_type                                                       |comment|
+-----------------------------+----------------------------------------------------------------+-------+
|category_name                |string                                                          |NULL   |
|month                        |string                                                          |NULL   |
|total_revenue                |decimal(14,2)                                                   |NULL   |
|order_count                  |bigint                                                          |NULL   |
|avg_check                    |decimal(12,2)                                                   |NULL   |
|                             |                                                           

In [26]:
print("[FORBIDDEN] INSERT в bronze.raw_orders (везде read-only для alex)")
try:
    spark.sql("""
        INSERT INTO lakehouse.bronze.raw_orders
        VALUES (99999, 1, 'completed', CAST(100.00 AS DECIMAL(12,2)), '2026-01-01')
    """)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

print("[FORBIDDEN] INSERT в silver.customers (read-only)")
try:
    spark.sql("""
        INSERT INTO lakehouse.silver.customers
        VALUES (99999, 'test', 'test@test.com', 'Moscow', CAST('2026-01-01' AS DATE))
    """)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

print("[FORBIDDEN] INSERT в gold.mart_top_customers (read-only)")
try:
    spark.sql("""
        INSERT INTO lakehouse.gold.mart_top_customers
        VALUES (99999, 'test', 1, CAST(1 AS BIGINT), CAST(100.00 AS DECIMAL(14,2)), 'Low')
    """)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

print("[FORBIDDEN] CREATE TABLE в bronze (нет TABLE_CREATE)")
try:
    spark.sql("""
        CREATE TABLE lakehouse.bronze._test_probe (
            id   INT,
            name STRING
        ) USING iceberg
    """)
    print("[ПРОВАЛ] Доступ разрешён — ожидалось запрещено")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Доступ запрещён: {short}")
    print()

[FORBIDDEN] INSERT в bronze.raw_orders (везде read-only для alex)
[ОЖИДАЕМО] Доступ запрещён: An error occurred while calling o43.sql.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'alex' with activated PrincipalRoles '[data_scientist]' and activated grants via '[data_scientist_catalog_role, data_scientist]' is not authorized for op ADD_TABLE_SNAPSHOT
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)

[FORBIDDEN] INSERT в silver.customers (read-only)
[ОЖИДАЕМО] Доступ запрещён: An error occurred while calling o43.sql.
: org.apache.iceberg.exceptions.ForbiddenException: Forbidden: Principal 'alex' with activated PrincipalRoles '[data_scientist]' and activated grants via '[data_scientist_catalog_role, data_scientist]' is not authorized for op ADD_TABLE_SNAPSHOT
	at org.apache.iceberg.rest.ErrorHandlers$DefaultErrorHandler.accept(ErrorHandlers.java:238)

[FORBIDDEN] INSERT в gold.mart_top_customers (read-only)
[ОЖИДАЕМО